# Sentiment Analysis Pipeline using Ollama
## *From Tweets to Trades: Analysing Sentiment Signal and Information Network for Crypto Market Prediction*

Runs LLM-based sentiment classification across all platforms (Twitter, Reddit, Bluesky, ProQuest news) and all asset classes (Bitcoin, Ethereum, Dogecoin, Shiba Inu, Tether) using a local Ollama model. Each platform-asset combination is processed independently, producing two outputs per combination:

1. **Per-post sentiment file** — every post tagged with its sentiment score and label.
2. **Daily-aggregated sentiment file** — sentiment metrics aggregated to one row per day, ready for time-series modelling against market data.

**Sentiment scale (5-category, mapped to -2 to +2):**
- Extremely Bearish (-2)
- Bearish (-1)
- Neutral (0)
- Bullish (+1)
- Extremely Bullish (+2)

**Daily aggregations computed (these become the predictor variables):**
- `sentiment_mean` — average sentiment for the day (primary signal).
- `sentiment_median` — robust central tendency.
- `sentiment_std` — within-day dispersion (proxy for disagreement).
- `post_count` — volume of activity (often predictive in itself).
- `bullish_ratio`, `bearish_ratio` — proportion of bullish vs bearish posts.
- `polarity_index` — bullish_ratio minus bearish_ratio (signed sentiment imbalance).
- `extreme_ratio` — fraction of posts at the extreme ends.

**Resume capability:** If the run is interrupted, restart this notebook and the processing will pick up from the last completed file.

## 0. Setup, Dependencies, and Ollama Installation

In [1]:
# Install zstd first (required by the Ollama installer).
!apt-get install -y zstd > /dev/null 2>&1

# Now install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in the background.
import subprocess
import time
ollama_proc = subprocess.Popen(['ollama', 'serve'],
                                stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

# Pull the model.
!ollama pull mistral
print('Model downloaded.')

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server started.

Model downloaded.


In [ ]:
# Check if the T4 GPU is visible to the system.
!nvidia-smi

# Check if Ollama can see it.
import subprocess
result = subprocess.run(['ollama', 'ps'], capture_output=True, text=True)
print("Ollama status:", result.stdout)

Thu Apr 16 10:42:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q ollama pandas openpyxl tqdm

import os
import json
import shutil
import time
import re
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm
from ollama import Client
from google.colab import files

# Working directories.
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('data/sentiment/per_post', exist_ok=True)
os.makedirs('data/sentiment/daily', exist_ok=True)
os.makedirs('data/sentiment/progress', exist_ok=True)
print('Setup complete.')

Setup complete.


In [ ]:
import time
from ollama import Client

client = Client()
start = time.time()
response = client.chat(
    model='mistral',
    messages=[{'role': 'user', 'content': 'Classify this as bullish or bearish: Bitcoin is going to moon!'}],
    options={'num_predict': 10}
)
elapsed = time.time() - start
print(f'Response time: {elapsed:.2f} seconds')
print(f'Response: {response["message"]["content"]}')

Response time: 103.48 seconds
Response:  The phrase "Bitcoin is going to moon!"


## 1. Configuration

Defines the model parameters (kept close to your sensitivity analysis defaults of `top_k=40, top_p=0.8, temperature=0.3` which gave stable results), the five-category sentiment scale, the prompt templates, and the asset detection rules.

In [ ]:
# --- Model configuration ---
MODEL_NAME = 'mistral'
MODEL_TOP_K = 40
MODEL_TOP_P = 0.8
MODEL_TEMP = 0.3

# --- Sentiment scale (matches your sensitivity analysis script) ---
LABEL_TO_SCORE = {
    'extremely bearish': -2,
    'bearish':           -1,
    'neutral':            0,
    'bullish':            1,
    'extremely bullish':  2,
}
SCORE_TO_LABEL = {v: k for k, v in LABEL_TO_SCORE.items()}

# --- Asset display names for prompts ---
ASSET_DISPLAY_NAMES = {
    'bitcoin':        'Bitcoin (BTC)',
    'ethereum':       'Ethereum (ETH)',
    'dogecoin':       'Dogecoin (DOGE)',
    'shiba_inu':      'Shiba Inu (SHIB)',
    'tether':         'Tether (USDT)',
    'crypto_general': 'cryptocurrency markets generally',
    'general':        'cryptocurrency markets generally',
}

# --- Filename detection patterns to extract platform and asset ---
PLATFORM_PATTERNS = {
    'twitter':  ['twitter', 'tweet', 'leoth9', 'infsceps', 'mendeley', 'merged_twitter'],
    'reddit':   ['reddit', 'rbitcoin', 'rdogecoin', 'rethereum', 'rshibarmy',
                 'rcryptocurrency', 'rsatoshistreetbets', 'rcryptomarkets'],
    'bluesky':  ['bluesky', 'bsky'],
    'news':     ['proquest', 'news', 'article', 'ft_', 'bloomberg'],
}

ASSET_PATTERNS = {
    'shiba_inu': ['shiba_inu', 'shibainu', 'shibarmy', 'shib_'],
    'bitcoin':   ['bitcoin', '_btc', 'rbitcoin'],
    'ethereum':  ['ethereum', '_eth', 'rethereum'],
    'dogecoin':  ['dogecoin', '_doge', 'rdogecoin'],
    'tether':    ['tether', 'usdt'],
    'crypto_general': ['crypto_general', 'rcryptocurrency',
                       'rcryptomarkets', 'general_'],
}

print(f'Model: {MODEL_NAME}')
print(f'Sampling: top_k={MODEL_TOP_K}, top_p={MODEL_TOP_P}, '
      f'temperature={MODEL_TEMP}')

Model: mistral
Sampling: top_k=40, top_p=0.8, temperature=0.3


In [ ]:
# --- Prompt templates ---
# Asset-aware system prompt: mentioning the specific asset improves
# classification accuracy because the model can distinguish posts
# about the asset from posts that mention it incidentally.

def build_system_prompt(asset_display_name):
    return f'''You are a financial markets sentiment analyst.
Given a single short text about {asset_display_name},
classify its sentiment regarding the asset into exactly one of these FIVE labels:
  - Extremely Bearish
  - Bearish
  - Neutral
  - Bullish
  - Extremely Bullish

Bullish means positive about price prospects.
Bearish means negative about price prospects.
Neutral means factual reporting or no clear directional view.
Use Extremely only when the text expresses strong conviction.

Respond with exactly one of those labels, and nothing else.'''

USER_TEMPLATE = '''Text:
"""
{txt}
"""

Label the sentiment according to the five-point scale above.'''

print('Prompt templates loaded.')

Prompt templates loaded.


## 2. Upload All Cleaned Datasets

Upload **all** the cleaned per-asset CSV files from every platform at once. This includes the cleaned outputs from the Twitter, Reddit, Bluesky, and ProQuest cleaning notebooks. The script will automatically detect which platform and asset each file belongs to from its filename.

In [ ]:
print('Upload all cleaned per-asset CSV files from every platform.')
print('You can drop the entire merged_twitter_corpus.csv plus all the')
print('per-asset Reddit, Bluesky, and ProQuest cleaned files at once.\n')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/cleaned/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content)/1e6:.2f} MB)')

print(f'\n{len(uploaded)} file(s) uploaded.')

Upload all cleaned per-asset CSV files from every platform.
You can drop the entire merged_twitter_corpus.csv plus all the
per-asset Reddit, Bluesky, and ProQuest cleaned files at once.



Saving bitcoin_bluesky_cleaned.csv to bitcoin_bluesky_cleaned.csv
  Saved: data/cleaned/bitcoin_bluesky_cleaned.csv (13.90 MB)

1 file(s) uploaded.


## 3. File Detection and Routing

Inspects each uploaded file and decides which platform-asset combination it belongs to. For multi-asset files (like the merged Twitter corpus), it splits the file by asset based on the `asset_mentions` column or the post text content.

In [ ]:
def detect_platform(filename):
    fname_lower = filename.lower()
    for platform, patterns in PLATFORM_PATTERNS.items():
        if any(p in fname_lower for p in patterns):
            return platform
    return None

def detect_asset(filename):
    fname_lower = filename.lower()
    for asset, patterns in ASSET_PATTERNS.items():
        if any(p in fname_lower for p in patterns):
            return asset
    return None

def detect_text_column(df):
    cols = {c.lower(): c for c in df.columns}
    for cand in ['text', 'full_text', 'selftext', 'body', 'content',
                 'title', '_combined']:
        if cand in cols:
            return cols[cand]
    return None

def detect_date_column(df):
    cols = {c.lower(): c for c in df.columns}
    for cand in ['date', 'created_at', 'pub_date', 'created_utc',
                 'timestamp']:
        if cand in cols:
            return cols[cand]
    return None

def split_multi_asset_file(df, text_col):
    """Split a multi-asset file (e.g., merged Twitter) into per-asset DataFrames."""
    asset_keywords = {
        'bitcoin':   ['bitcoin', 'btc'],
        'ethereum':  ['ethereum', 'ether', 'eth'],
        'dogecoin':  ['dogecoin', 'doge'],
        'shiba_inu': ['shiba', 'shib'],
        'tether':    ['tether', 'usdt'],
    }
    splits = {}
    # If asset_mentions column exists (created by the Twitter merge script),
    # use that. Otherwise, do keyword matching on text.
    if 'asset_mentions' in df.columns:
        for asset in asset_keywords:
            mask = df['asset_mentions'].str.contains(asset, na=False)
            if mask.any():
                splits[asset] = df[mask].copy()
    else:
        text_lower = df[text_col].astype(str).str.lower()
        for asset, kws in asset_keywords.items():
            pattern = '|'.join(re.escape(k) for k in kws)
            mask = text_lower.str.contains(pattern, na=False, regex=True)
            if mask.any():
                splits[asset] = df[mask].copy()
    return splits

# Build the processing job list.
INPUT_DIR = 'data/cleaned'
csv_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')])

print(f'Detected {len(csv_files)} input file(s):\n')
jobs = []  # list of (job_name, df, text_col, date_col, asset, platform)

for fname in csv_files:
    platform = detect_platform(fname)
    asset = detect_asset(fname)
    print(f'  {fname}')
    print(f'    Platform: {platform} | Asset: {asset}')

    if not platform:
        print(f'    SKIPPED: could not identify platform.\n')
        continue

    df = pd.read_csv(os.path.join(INPUT_DIR, fname),
                     on_bad_lines='skip', engine='python',
                     encoding='utf-8', encoding_errors='ignore')
    text_col = detect_text_column(df)
    date_col = detect_date_column(df)

    if not text_col:
        print(f'    SKIPPED: no text column found.\n')
        continue

    # If asset is unknown, this is likely a multi-asset file: split it.
    if not asset:
        print(f'    Multi-asset file: splitting by asset...')
        splits = split_multi_asset_file(df, text_col)
        for asset_name, asset_df in splits.items():
            job_name = f'{platform}_{asset_name}'
            jobs.append((job_name, asset_df, text_col, date_col,
                         asset_name, platform))
            print(f'      {asset_name}: {len(asset_df):,} posts')
    else:
        job_name = f'{platform}_{asset}'
        jobs.append((job_name, df, text_col, date_col, asset, platform))
        print(f'    Rows: {len(df):,}\n')

print(f'\n{len(jobs)} sentiment analysis job(s) queued.')

Detected 1 input file(s):

  bitcoin_bluesky_cleaned.csv
    Platform: bluesky | Asset: bitcoin
    Rows: 39,113


1 sentiment analysis job(s) queued.


## 3b. Stratified Sampling (100 posts per day per platform-asset)

Caps the number of posts per day per platform-asset combination to keep runtime manageable while preserving statistically valid daily sentiment estimates. Days with fewer than the cap keep all their posts. Days with more are randomly subsampled using a fixed seed for reproducibility.

**Methodological rationale:** Daily sentiment is the unit of analysis for the time-series modelling. Standard sampling theory shows that the standard error of a daily mean decreases with the square root of sample size, so returns diminish rapidly beyond roughly 100 posts per day. Capping at this threshold preserves essentially all statistical power while substantially reducing computational cost.

In [ ]:
# --- Stratified sampling configuration ---
POSTS_PER_DAY_CAP = 100
RANDOM_SEED = 42  # For reproducibility.

def stratified_sample_by_day(df, date_col, cap=POSTS_PER_DAY_CAP,
                               seed=RANDOM_SEED):
    """
    Cap posts per day at `cap`. Days with fewer than `cap` posts are
    kept in full; days with more are randomly sampled down to `cap`.
    A fixed seed ensures the same sample is drawn on every run.
    """
    if date_col not in df.columns or len(df) == 0:
        return df

    # Parse dates to a clean YYYY-MM-DD string for grouping.
    parsed = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    if parsed.dt.tz is not None:
        parsed = parsed.dt.tz_localize(None)
    df = df.copy()
    df['_sample_day'] = parsed.dt.strftime('%Y-%m-%d')

    # Drop rows with unparseable dates (they cannot be aggregated
    # into a daily panel anyway).
    df = df[df['_sample_day'].notna()]

    # Sample within each day.
    sampled = (
        df.groupby('_sample_day', group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), cap),
                                     random_state=seed))
    )
    sampled = sampled.drop(columns=['_sample_day']).reset_index(drop=True)
    return sampled

# Apply sampling to each queued job and build a new job list.
print(f'Applying stratified sampling (cap = {POSTS_PER_DAY_CAP} '
      f'posts per day, seed = {RANDOM_SEED})\n')
print('=' * 70)

sampled_jobs = []
total_before = 0
total_after = 0

for job_name, df, text_col, date_col, asset, platform in jobs:
    before = len(df)
    total_before += before

    if date_col and before > 0:
        df_sampled = stratified_sample_by_day(df, date_col)
    else:
        print(f'  {job_name}: no date column, keeping all posts.')
        df_sampled = df

    after = len(df_sampled)
    total_after += after
    reduction = 100 * (1 - after / before) if before > 0 else 0
    print(f'  {job_name:35s} {before:>8,} -> {after:>8,} '
          f'({reduction:5.1f}% reduction)')

    sampled_jobs.append((job_name, df_sampled, text_col, date_col,
                          asset, platform))

# Replace the jobs list with the sampled version.
jobs = sampled_jobs

print('=' * 70)
print(f'Total posts before sampling: {total_before:>10,}')
print(f'Total posts after sampling:  {total_after:>10,}')
if total_before > 0:
    print(f'Overall reduction:           {100 * (1 - total_after / total_before):>9.1f}%')
print(f'\nEstimated runtime at 1 sec/post: '
      f'{total_after / 3600:.1f} hours')

Applying stratified sampling (cap = 100 posts per day, seed = 42)

  bluesky_bitcoin                       39,113 ->   28,239 ( 27.8% reduction)
Total posts before sampling:     39,113
Total posts after sampling:      28,239
Overall reduction:                27.8%

Estimated runtime at 1 sec/post: 7.8 hours


/tmp/ipykernel_9844/2297008932.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), cap),


## 4. Sentiment Classification Function

Calls Ollama for each post. Includes retry logic for transient errors and a fallback parser that handles slight variations in the model's output (case differences, leading/trailing whitespace, prefixed reasoning text).

In [ ]:
_client = Client()

def classify_text(text, asset_display_name, max_retries=2):
    """Classify a single post into the five-point sentiment scale."""
    if not text or not isinstance(text, str) or len(text.strip()) < 5:
        return None

    # Truncate very long texts (especially news articles) to avoid slow
    # processing. The first 1500 chars contain the lead and primary tone.
    text = text[:1500].strip()

    system_prompt = build_system_prompt(asset_display_name)
    user_prompt = USER_TEMPLATE.format(txt=text)

    for attempt in range(max_retries + 1):
        try:
            response = _client.chat(
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt},
                ],
                options={
                    'top_k': MODEL_TOP_K,
                    'top_p': MODEL_TOP_P,
                    'temperature': MODEL_TEMP,
                    'num_predict': 30,  # Cap output length to speed up.
                }
            )
            raw = response['message']['content'].strip().lower()
            # Try exact prefix match first (most reliable).
            for label, score in LABEL_TO_SCORE.items():
                if raw.startswith(label):
                    return score
            # Fallback: substring match anywhere in response.
            # Check longer labels first to avoid 'bearish' matching
            # 'extremely bearish'.
            for label in sorted(LABEL_TO_SCORE.keys(),
                                 key=len, reverse=True):
                if label in raw:
                    return LABEL_TO_SCORE[label]
            return None  # Unparseable response.
        except Exception as e:
            if attempt == max_retries:
                return None
            time.sleep(1)
    return None

# Quick smoke test.
test_score = classify_text(
    'Bitcoin just hit a new all-time high, the bull run is unstoppable!',
    'Bitcoin (BTC)'
)
print(f'Smoke test: "Bitcoin just hit a new all-time high" -> '
      f'{test_score} ({SCORE_TO_LABEL.get(test_score, "unparsed")})')

Smoke test: "Bitcoin just hit a new all-time high" -> 1 (bullish)


## 5. Daily Aggregation Function

Collapses per-post sentiment scores into one row per day, computing all the metrics needed for time-series modelling against market data.

In [ ]:
def aggregate_daily(df, date_col, score_col='sentiment_score'):
    """Collapse per-post scores into a daily panel for time-series analysis."""
    valid = df[df[score_col].notna()].copy()
    if len(valid) == 0:
        return pd.DataFrame()

    # Standardise the date to YYYY-MM-DD.
    valid['_date'] = pd.to_datetime(valid[date_col], errors='coerce',
                                      utc=True)
    if valid['_date'].dt.tz is not None:
        valid['_date'] = valid['_date'].dt.tz_localize(None)
    valid['_date'] = valid['_date'].dt.strftime('%Y-%m-%d')
    valid = valid[valid['_date'].notna()]

    # Group by date and compute the panel of sentiment metrics.
    grouped = valid.groupby('_date')[score_col]
    daily = pd.DataFrame({
        'date':            grouped.first().index,
        'post_count':      grouped.count().values,
        'sentiment_mean':  grouped.mean().values,
        'sentiment_median': grouped.median().values,
        'sentiment_std':   grouped.std().values,
    })

    # Categorical breakdowns.
    bullish_counts = valid[valid[score_col] > 0].groupby('_date').size()
    bearish_counts = valid[valid[score_col] < 0].groupby('_date').size()
    neutral_counts = valid[valid[score_col] == 0].groupby('_date').size()
    extreme_counts = valid[valid[score_col].abs() == 2].groupby('_date').size()

    daily['bullish_count'] = daily['date'].map(bullish_counts).fillna(0).astype(int)
    daily['bearish_count'] = daily['date'].map(bearish_counts).fillna(0).astype(int)
    daily['neutral_count'] = daily['date'].map(neutral_counts).fillna(0).astype(int)
    daily['extreme_count'] = daily['date'].map(extreme_counts).fillna(0).astype(int)

    # Ratios (proportional metrics that are scale-independent).
    daily['bullish_ratio'] = daily['bullish_count'] / daily['post_count']
    daily['bearish_ratio'] = daily['bearish_count'] / daily['post_count']
    daily['neutral_ratio'] = daily['neutral_count'] / daily['post_count']
    daily['extreme_ratio'] = daily['extreme_count'] / daily['post_count']

    # Polarity index: the signed imbalance between bullish and bearish.
    daily['polarity_index'] = daily['bullish_ratio'] - daily['bearish_ratio']

    # Sort chronologically.
    daily = daily.sort_values('date').reset_index(drop=True)
    return daily

print('Daily aggregation function loaded.')

Daily aggregation function loaded.


## 6. Run Sentiment Analysis on All Jobs

Processes every queued platform-asset combination. Each job saves a per-post CSV (with sentiment columns added to the original data) and a daily-aggregated CSV (ready for time-series modelling). Resume capability is built in: if you re-run this cell after an interruption, completed jobs are skipped and incomplete jobs pick up from the last saved batch.

**Save frequency:** every 50 posts to minimise data loss if interrupted.

In [ ]:
PER_POST_DIR = 'data/sentiment/per_post'
DAILY_DIR = 'data/sentiment/daily'
PROGRESS_DIR = 'data/sentiment/progress'
SAVE_EVERY = 50  # Save partial results every N posts.

def run_job(job_name, df, text_col, date_col, asset, platform):
    print(f'\n{"=" * 60}')
    print(f'JOB: {job_name}')
    print(f'{"=" * 60}')
    print(f'  Platform: {platform} | Asset: {asset}')
    print(f'  Total posts: {len(df):,}')

    per_post_path = os.path.join(PER_POST_DIR, f'{job_name}_sentiment.csv')
    daily_path = os.path.join(DAILY_DIR, f'{job_name}_daily.csv')
    progress_path = os.path.join(PROGRESS_DIR, f'{job_name}_progress.json')

    # Resume logic: load any existing progress.
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            progress = json.load(f)
        if progress.get('completed', False):
            print(f'  Already completed. Skipping. '
                  f'(Delete {progress_path} to redo.)')
            return
        last_idx = progress.get('last_idx', -1)
    else:
        last_idx = -1

    # Load any partial per-post results.
    if os.path.exists(per_post_path) and last_idx >= 0:
        partial = pd.read_csv(per_post_path)
        scores_dict = dict(zip(partial.index, partial['sentiment_score']))
        print(f'  Resuming from index {last_idx + 1} '
              f'({len(scores_dict)} already classified).')
    else:
        scores_dict = {}

    # Asset display name for the prompt.
    asset_display = ASSET_DISPLAY_NAMES.get(asset, asset.title())

    # Iterate through posts.
    df = df.reset_index(drop=True)
    df['sentiment_score'] = df.index.map(scores_dict)

    indices_to_process = [i for i in df.index if i > last_idx
                           and i not in scores_dict]
    if not indices_to_process:
        print('  Nothing to process. All posts already classified.')
    else:
        for count, idx in enumerate(tqdm(indices_to_process,
                                          desc=job_name)):
            text = df.at[idx, text_col]
            score = classify_text(text, asset_display)
            df.at[idx, 'sentiment_score'] = score

            # Periodic save.
            if (count + 1) % SAVE_EVERY == 0:
                df.to_csv(per_post_path, index=False)
                with open(progress_path, 'w') as f:
                    json.dump({'last_idx': int(idx),
                                'completed': False}, f)

    # Add the human-readable label column.
    df['sentiment_label'] = df['sentiment_score'].map(SCORE_TO_LABEL)

    # Final save of per-post file.
    df.to_csv(per_post_path, index=False)

    # Compute daily aggregation.
    if date_col:
        daily = aggregate_daily(df, date_col)
        if len(daily) > 0:
            daily['asset'] = asset
            daily['platform'] = platform
            daily.to_csv(daily_path, index=False)
            print(f'\n  Per-post saved: {per_post_path} ({len(df):,} rows)')
            print(f'  Daily aggregated: {daily_path} ({len(daily):,} days)')
            print(f'  Date range: {daily["date"].min()} to '
                  f'{daily["date"].max()}')
            print(f'  Mean sentiment: {daily["sentiment_mean"].mean():.3f}')
            print(f'  Mean polarity: {daily["polarity_index"].mean():.3f}')
        else:
            print(f'  Warning: no valid dates for daily aggregation.')
    else:
        print(f'  Warning: no date column, skipping daily aggregation.')

    # Mark job as completed.
    with open(progress_path, 'w') as f:
        json.dump({'last_idx': len(df) - 1, 'completed': True}, f)

# Run all jobs sequentially.
for job in jobs:
    run_job(*job)

print('\n' + '=' * 60)
print('ALL JOBS COMPLETE')
print('=' * 60)


JOB: bluesky_bitcoin
  Platform: bluesky | Asset: bitcoin
  Total posts: 28,239


bluesky_bitcoin:   0%|          | 0/28239 [00:00<?, ?it/s]


  Per-post saved: data/sentiment/per_post/bluesky_bitcoin_sentiment.csv (28,239 rows)
  Daily aggregated: data/sentiment/daily/bluesky_bitcoin_daily.csv (607 days)
  Date range: 2023-03-01 to 2024-12-30
  Mean sentiment: 0.101
  Mean polarity: 0.098

ALL JOBS COMPLETE


## 7. Build Combined Excel Workbook for Time-Series Modelling

Bundles all the daily aggregated sentiment data into a single Excel workbook with one sheet per platform-asset combination, plus an index sheet listing every dataset and a combined long-format sheet that has all platform-asset rows stacked together. This format is ideal for downstream analysis: load the workbook, read the asset sheet you need, merge with the cleaned market data on the date column, and proceed with VAR/Granger/transfer entropy.

In [ ]:
WORKBOOK_PATH = 'sentiment_daily_panel.xlsx'

daily_files = sorted([f for f in os.listdir(DAILY_DIR)
                       if f.endswith('_daily.csv')])

if not daily_files:
    print('No daily aggregated files found. Run Section 6 first.')
else:
    all_daily = []
    index_rows = []

    with pd.ExcelWriter(WORKBOOK_PATH, engine='openpyxl') as writer:
        for fname in daily_files:
            sheet_name = fname.replace('_daily.csv', '')[:31]  # Excel cap
            df = pd.read_csv(os.path.join(DAILY_DIR, fname))
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            all_daily.append(df)
            index_rows.append({
                'sheet_name': sheet_name,
                'platform': df['platform'].iloc[0] if 'platform' in df.columns else '',
                'asset': df['asset'].iloc[0] if 'asset' in df.columns else '',
                'days': len(df),
                'total_posts': int(df['post_count'].sum()),
                'date_min': df['date'].min(),
                'date_max': df['date'].max(),
                'mean_sentiment': round(df['sentiment_mean'].mean(), 3),
            })

        # Combined long-format sheet.
        if all_daily:
            combined = pd.concat(all_daily, ignore_index=True)
            combined = combined.sort_values(['asset', 'platform', 'date'])
            combined.to_excel(writer, sheet_name='_combined_all',
                              index=False)

        # Index sheet (placed first when opened).
        index_df = pd.DataFrame(index_rows).sort_values(
            ['asset', 'platform'])
        index_df.to_excel(writer, sheet_name='_index', index=False)

    print(f'Workbook saved: {WORKBOOK_PATH}')
    print(f'Sheets: {len(daily_files) + 2} '
          f'({len(daily_files)} datasets + index + combined)')
    print(f'\nIndex preview:')
    print(index_df.to_string(index=False))

Workbook saved: sentiment_daily_panel.xlsx
Sheets: 3 (1 datasets + index + combined)

Index preview:
     sheet_name platform   asset  days  total_posts   date_min   date_max  mean_sentiment
bluesky_bitcoin  bluesky bitcoin   607        28239 2023-03-01 2024-12-30           0.101


## 8. Download All Results

In [ ]:
# Bundle everything: the workbook, the per-post CSVs, and the daily CSVs.
shutil.make_archive('sentiment_results', 'zip',
                    'data/sentiment')

# Also copy the workbook into the archive folder for convenience.
if os.path.exists(WORKBOOK_PATH):
    shutil.copy(WORKBOOK_PATH, 'data/sentiment/sentiment_daily_panel.xlsx')
    shutil.make_archive('sentiment_results', 'zip',
                        'data/sentiment')

size_mb = os.path.getsize('sentiment_results.zip') / 1e6
print(f'Archive: sentiment_results.zip ({size_mb:.2f} MB)')
files.download('sentiment_results.zip')

# Also offer the workbook on its own for convenience.
if os.path.exists(WORKBOOK_PATH):
    files.download(WORKBOOK_PATH)

print('Downloads started.')

Archive: sentiment_results.zip (3.00 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started.
